# SAE-TCAV robustness analysis

Post-hoc analysis of completed comparison artifacts. This notebook performs no
training, inference, resampling, or source-artifact mutation. It freezes SAE run
0 as canonical, builds overlapping factor subsets, measures pooled/class temporal
robustness, and summarizes pooled/class semantic transfer.

Run from repository root with Python 3.11 and packages in
`requirements-semantic.txt`. Edit only configuration values below.



In [ ]:
from __future__ import annotations

from collections import Counter
from hashlib import sha256
import json
import math
import os
from pathlib import Path
import pickle
import tempfile
from typing import Any, Iterable, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

STATS_ROOT = Path("stats")
RUNNER_DIR: Path | None = None
RUNNER_HASH: str | None = None
CANONICAL_RUN = 0
ACTIVATION_EPSILON = 1e-5
ROBUST_F2_MINIMUM = 0.70
OUTCOME_CLASSES = (0, 1)

SAVE_PLOTS = True
DISPLAY_PLOTS = True
PLOT_FORMAT = "png"
FIGURE_DPI = 160

SUBSET_ORDER = ("all", "non_dead", "precision_recall", "f2", "robust")
SUBSET_LABELS = {
    "all": "All factors",
    "non_dead": "Non-dead",
    "precision_recall": "Precision/recall",
    "f2": "F2",
    "robust": "Robust",
}

RUNNER_REQUIRED = (
    "summary.json",
    "runner_manifest.json",
    "sae_manifest.json",
    "semantic_inputs.npz",
    "activations.npz",
    "prepared.pkl",
    "matched_factors.csv",
    "high_precision_rules.jsonl",
    "functional.json",
    "tabpfn_metrics.json",
)
SEMANTIC_REQUIRED = (
    "manifest.json",
    "semantic_rules.jsonl",
    "pair_results.jsonl",
    "pair_metrics.csv",
    "pair_metrics_by_class.csv",
)

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.22,
    }
)



## Artifact discovery and strict loading

Absolute paths stored by the producing machine are treated as hints. Relocatable
fallbacks use the runner hash and semantic experiment hash.



In [ ]:
def read_json(path: Path) -> Any:
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def array_fingerprint(array: np.ndarray) -> str:
    value = np.asarray(array)
    digest = sha256()
    digest.update(str(value.dtype).encode())
    digest.update(json.dumps(value.shape).encode())
    if value.dtype.kind in {"O", "U", "S"}:
        digest.update(json.dumps(value.tolist(), separators=(",", ":")).encode())
    else:
        digest.update(np.ascontiguousarray(value).tobytes())
    return digest.hexdigest()


def require_files(root: Path, names: Sequence[str], label: str) -> None:
    missing = [name for name in names if not (root / name).is_file()]
    if missing:
        raise FileNotFoundError(f"{label} missing required artifacts: {missing}")


def semantic_dir_candidates(
    runner_dir: Path, summary: Mapping[str, Any], stats_root: Path
) -> list[Path]:
    experiment_hash = str(summary.get("semantic_experiment_hash", ""))
    raw = summary.get("semantic_artifact_dir")
    candidates: list[Path] = []
    if raw:
        raw_path = Path(str(raw))
        candidates.extend(
            [
                raw_path,
                Path.cwd() / raw_path,
                runner_dir / raw_path,
                stats_root / "semantic" / raw_path.name,
            ]
        )
    if experiment_hash:
        candidates.extend(
            [
                runner_dir / "semantic" / experiment_hash,
                stats_root / "semantic" / experiment_hash,
                runner_dir.parent.parent / "semantic" / experiment_hash,
            ]
        )
    unique: list[Path] = []
    seen: set[str] = set()
    for candidate in candidates:
        key = str(candidate.resolve(strict=False))
        if key not in seen:
            unique.append(candidate)
            seen.add(key)
    return unique


def resolve_semantic_dir(
    runner_dir: Path, summary: Mapping[str, Any], stats_root: Path
) -> Path:
    for candidate in semantic_dir_candidates(runner_dir, summary, stats_root):
        if candidate.is_dir() and all((candidate / name).is_file() for name in SEMANTIC_REQUIRED):
            return candidate
    tried = [str(path) for path in semantic_dir_candidates(runner_dir, summary, stats_root)]
    raise FileNotFoundError(f"No complete semantic artifact directory. Tried: {tried}")


def is_complete_runner(path: Path, stats_root: Path) -> bool:
    try:
        require_files(path, RUNNER_REQUIRED, "runner")
        resolve_semantic_dir(path, read_json(path / "summary.json"), stats_root)
    except (OSError, ValueError, TypeError, KeyError, json.JSONDecodeError):
        return False
    return True


def discover_runner(
    stats_root: Path,
    runner_dir: Path | None = None,
    runner_hash: str | None = None,
) -> tuple[Path, Path, dict[str, Any]]:
    if runner_dir is not None and runner_hash is not None:
        raise ValueError("Set RUNNER_DIR or RUNNER_HASH, not both")
    comparison_root = stats_root / "comparison"
    if runner_dir is not None:
        candidates = [Path(runner_dir)]
    elif runner_hash is not None:
        candidates = [comparison_root / runner_hash]
    else:
        candidates = sorted(
            (
                path
                for path in comparison_root.iterdir()
                if path.is_dir() and path.name != "_cache"
            ),
            key=lambda path: (path / "summary.json").stat().st_mtime
            if (path / "summary.json").is_file()
            else -1,
            reverse=True,
        )
    for candidate in candidates:
        if is_complete_runner(candidate, stats_root):
            summary = read_json(candidate / "summary.json")
            return candidate, resolve_semantic_dir(candidate, summary, stats_root), summary
    raise FileNotFoundError(
        f"No complete runner found among {[str(path) for path in candidates]}"
    )


def prepared_value(prepared: Any, name: str) -> Any:
    if isinstance(prepared, Mapping):
        return prepared[name]
    return getattr(prepared, name)


def load_npz(path: Path, prefix: str) -> dict[int, np.ndarray]:
    with np.load(path, allow_pickle=False) as archive:
        values = {
            int(key.removeprefix(prefix)): np.asarray(archive[key])
            for key in archive.files
            if key.startswith(prefix)
        }
    if not values:
        raise ValueError(f"{path} contains no {prefix}<run_id> arrays")
    return values


def require_columns(frame: pd.DataFrame, columns: Iterable[str], label: str) -> None:
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{label} missing columns: {missing}")


def read_csv_or_empty(path: Path, columns: Sequence[str]) -> pd.DataFrame:
    if path.stat().st_size == 0:
        return pd.DataFrame(columns=list(columns))
    return pd.read_csv(path)


def match_fields(row: Mapping[str, Any]) -> tuple[int, int, int, int]:
    if all(key in row for key in ("sae_i_idx", "sae_j_idx", "original_concept", "best_pair")):
        return (
            int(row["sae_i_idx"]),
            int(row["sae_j_idx"]),
            int(row["original_concept"]),
            int(row["best_pair"]),
        )
    return (
        int(row["run_i"]),
        int(row["run_j"]),
        int(row["factor_i"]),
        int(row["factor_j"]),
    )


def canonical_factor(row: Mapping[str, Any], canonical_run: int) -> int | None:
    run_i, run_j, factor_i, factor_j = match_fields(row)
    if run_i == canonical_run:
        return factor_i
    if run_j == canonical_run:
        return factor_j
    return None


def direct_other_run(row: Mapping[str, Any], canonical_run: int) -> int | None:
    run_i, run_j, _, _ = match_fields(row)
    if run_i == canonical_run:
        return run_j
    if run_j == canonical_run:
        return run_i
    return None


def load_artifacts(
    stats_root: Path,
    runner_dir: Path | None,
    runner_hash: str | None,
    canonical_run: int,
) -> dict[str, Any]:
    runner, semantic, summary = discover_runner(stats_root, runner_dir, runner_hash)
    require_files(runner, RUNNER_REQUIRED, "runner")
    require_files(semantic, SEMANTIC_REQUIRED, "semantic")

    runner_manifest = read_json(runner / "runner_manifest.json")
    sae_manifest = read_json(runner / "sae_manifest.json")
    semantic_manifest = read_json(semantic / "manifest.json")
    if summary["runner_hash"] != runner_manifest["runner_hash"]:
        raise ValueError("summary and runner manifest hashes differ")
    if runner.name != str(summary["runner_hash"]):
        raise ValueError("runner directory name differs from runner hash")
    if str(summary["semantic_experiment_hash"]) != str(semantic_manifest["experiment_hash"]):
        raise ValueError("semantic experiment hashes differ")
    if semantic_manifest["config"]["objective"]["objective"] != "f2":
        raise ValueError("semantic selection objective must equal 'f2'")
    if not bool(semantic_manifest["config"]["class_analysis"]["enabled"]):
        raise ValueError("class analysis must be enabled")

    with np.load(runner / "semantic_inputs.npz", allow_pickle=False) as bundle:
        required_arrays = {"X", "outcome", "patient_ids", "feature_names", "record_keys"}
        missing = required_arrays - set(bundle.files)
        if missing:
            raise ValueError(f"semantic_inputs.npz missing arrays: {sorted(missing)}")
        X = np.asarray(bundle["X"])
        outcome = np.asarray(bundle["outcome"])
        patient_ids = np.asarray(bundle["patient_ids"]).astype(str)
        feature_names = np.asarray(bundle["feature_names"]).astype(str)
        record_keys = np.asarray(bundle["record_keys"]).astype(str)
        activations = {
            int(key.removeprefix("activations_run_")): np.asarray(bundle[key])
            for key in bundle.files
            if key.startswith("activations_run_")
        }
    activation_copy = load_npz(runner / "activations.npz", "run_")
    run_ids = {int(row["run_id"]) for row in sae_manifest}
    if set(activations) != run_ids or set(activation_copy) != run_ids:
        raise ValueError("activation run IDs and SAE manifest differ")
    if canonical_run not in run_ids:
        raise ValueError(f"canonical run {canonical_run} unavailable")

    n_rows = len(X)
    if not (
        len(outcome) == len(patient_ids) == len(record_keys) == n_rows
        == int(runner_manifest["n_test_records"])
        == int(semantic_manifest["n_records"])
    ):
        raise ValueError("row counts differ across bundle and manifests")
    if X.ndim != 2 or X.shape[1] != len(feature_names):
        raise ValueError("X and feature names differ")
    for run_id in sorted(run_ids):
        first, second = activations[run_id], activation_copy[run_id]
        expected_factors = int(next(row["n_factors"] for row in sae_manifest if int(row["run_id"]) == run_id))
        if first.shape != (n_rows, expected_factors):
            raise ValueError(f"run {run_id} activation shape differs from manifest")
        if first.shape != second.shape or not np.array_equal(first, second, equal_nan=True):
            raise ValueError(f"run {run_id} activation archives differ")
        expected_hash = semantic_manifest["activation_fingerprints"][str(run_id)]
        if array_fingerprint(first) != expected_hash:
            raise ValueError(f"run {run_id} activation fingerprint differs")
        if not np.isfinite(first).all():
            raise ValueError(f"run {run_id} activations contain non-finite values")
    for value, key in (
        (X, "data_fingerprint"),
        (outcome, "outcome_fingerprint"),
        (patient_ids, "patient_group_fingerprint"),
    ):
        if array_fingerprint(value) != semantic_manifest[key]:
            raise ValueError(f"{key} differs")

    with (runner / "prepared.pkl").open("rb") as handle:
        prepared = pickle.load(handle)
    years = np.asarray(prepared_value(prepared, "years_test"), dtype=int)
    prepared_outcome = np.asarray(prepared_value(prepared, "y_test"))
    prepared_X = np.asarray(prepared_value(prepared, "X_test"))
    prepared_keys = np.asarray(prepared_value(prepared, "record_keys")).astype(str)
    if not (
        np.array_equal(years.shape, outcome.shape)
        and np.array_equal(prepared_outcome, outcome)
        and np.array_equal(prepared_X, X, equal_nan=True)
        and np.array_equal(prepared_keys, record_keys)
    ):
        raise ValueError("prepared.pkl rows do not align with semantic bundle")
    if array_fingerprint(np.asarray(prepared_value(prepared, "record_keys"))) != runner_manifest["record_fingerprint"]:
        raise ValueError("prepared record fingerprint differs from runner manifest")

    match_columns = (
        "sae_i_idx", "sae_j_idx", "original_concept", "best_pair",
        "cos_sim", "overlap",
    )
    matches = read_csv_or_empty(runner / "matched_factors.csv", match_columns)
    require_columns(
        matches,
        ("sae_i_idx", "sae_j_idx", "original_concept", "best_pair", "cos_sim"),
        "matched_factors.csv",
    )
    high_precision = read_jsonl(runner / "high_precision_rules.jsonl")
    functional = read_json(runner / "functional.json")
    semantic_rules = read_jsonl(semantic / "semantic_rules.jsonl")
    pair_results = read_jsonl(semantic / "pair_results.jsonl")
    pair_required = (
        "pair_id", "run_i", "run_j", "factor_i", "factor_j", "threshold_name",
        "positive_fraction", "i_to_j_precision", "i_to_j_recall", "i_to_j_f2",
        "j_to_i_precision", "j_to_i_recall", "j_to_i_f2",
        "transfer_mean_f2", "transfer_min_f2", "target_cohort_jaccard",
        "selected_cohort_jaccard", "cos_sim",
    )
    class_required = pair_required + (
        "class_value", "class_n_samples", "class_n_positive_i",
        "class_n_positive_j", "class_valid", "class_reasons",
    )
    pair_metrics = read_csv_or_empty(semantic / "pair_metrics.csv", pair_required)
    pair_metrics_class = read_csv_or_empty(
        semantic / "pair_metrics_by_class.csv", class_required
    )
    tabpfn_metrics = pd.DataFrame(read_json(runner / "tabpfn_metrics.json"))
    require_columns(pair_metrics, pair_required, "pair_metrics.csv")
    require_columns(pair_metrics_class, class_required, "pair_metrics_by_class.csv")
    expected_thresholds = {
        row["target"]["name"]
        for row in semantic_rules
        if str(row["run_id"]) == str(canonical_run)
    }
    configured_fractions = semantic_manifest["config"]["activation_targets"]["positive_fractions"]
    configured_names = {
        f"top_{str(int(100 * value)) if float(100 * value).is_integer() else str(100 * value).replace('.', '_')}pct_positive"
        for value in configured_fractions
    }
    if expected_thresholds and expected_thresholds != configured_names:
        raise ValueError("semantic rule threshold names differ from configured activation targets")
    if len(pair_metrics) and set(pair_metrics["threshold_name"].astype(str)) != configured_names:
        raise ValueError("pooled pair threshold names differ from configuration")
    if len(pair_metrics_class) and set(pair_metrics_class["threshold_name"].astype(str)) != configured_names:
        raise ValueError("class pair threshold names differ from configuration")

    if len(matches) != int(runner_manifest["n_selected_matches"]):
        raise ValueError("matched factor count differs from runner manifest")
    if len(matches) != int(summary["n_selected_matches"]):
        raise ValueError("matched factor count differs from summary")
    bounds = {int(item["run_id"]): int(item["n_factors"]) for item in sae_manifest}
    for row in matches.to_dict("records"):
        run_i, run_j, factor_i, factor_j = match_fields(row)
        if not (0 <= factor_i < bounds[run_i] and 0 <= factor_j < bounds[run_j]):
            raise ValueError("matched factor outside SAE bounds")
    for row in semantic_rules:
        run_id, factor_id = int(row["run_id"]), int(row["factor_id"])
        bound = int(next(item["n_factors"] for item in sae_manifest if int(item["run_id"]) == run_id))
        if not 0 <= factor_id < bound:
            raise ValueError("semantic factor outside SAE bounds")
        if row["selection"]["rule_set"]["threshold_name"] != row["target"]["name"]:
            raise ValueError("semantic rule threshold names are inconsistent")
    for row in high_precision:
        run_id, factor_id = int(row["run_id"]), int(row["Factor"])
        if run_id not in bounds or not 0 <= factor_id < bounds[run_id]:
            raise ValueError("high-precision rule factor outside SAE bounds")
        if row.get("Provenance") not in {"high_precision", "forced_fallback"}:
            raise ValueError("unknown high-precision rule provenance")

    pooled_support = pair_metrics.groupby(["pair_id", "threshold_name"], dropna=False).size()
    if not (pooled_support == 1).all():
        raise ValueError("pooled pair metrics must contain one row per pair/threshold")
    expected_pooled_rows = sum(len(pair["thresholds"]) for pair in pair_results)
    expected_class_rows = sum(
        len(threshold.get("class_analysis", []))
        for pair in pair_results
        for threshold in pair["thresholds"]
    )
    if len(pair_metrics) != expected_pooled_rows:
        raise ValueError("pooled CSV row count differs from pair results")
    if len(pair_metrics_class) != expected_class_rows:
        raise ValueError("class CSV row count differs from pair results")
    class_sums = pair_metrics_class.groupby(["pair_id", "threshold_name"])["class_n_samples"].sum()
    pooled_n = {
        (int(pair["pair_id"]), str(threshold["threshold_name"])): int(threshold["transfer"]["n_samples"])
        for pair in pair_results
        for threshold in pair["thresholds"]
    }
    for key, support in class_sums.items():
        if int(support) != pooled_n[(int(key[0]), str(key[1]))]:
            raise ValueError(f"class support does not sum to pooled support for {key}")
    manifest_support = {
        str(key): int(value)
        for key, value in semantic_manifest["class_analysis"]["class_support"].items()
    }
    for (_, threshold), group in pair_metrics_class.groupby(["pair_id", "threshold_name"]):
        observed = {
            str(row.class_value): int(row.class_n_samples)
            for row in group.itertuples()
        }
        if observed != manifest_support:
            raise ValueError(f"class support differs from manifest at threshold {threshold}")

    return {
        "runner_dir": runner,
        "semantic_dir": semantic,
        "summary": summary,
        "runner_manifest": runner_manifest,
        "sae_manifest": sae_manifest,
        "semantic_manifest": semantic_manifest,
        "X": X,
        "outcome": outcome,
        "patient_ids": patient_ids,
        "feature_names": feature_names,
        "record_keys": record_keys,
        "years": years,
        "activations": activations,
        "matches": matches,
        "high_precision": high_precision,
        "functional": functional,
        "semantic_rules": semantic_rules,
        "pair_results": pair_results,
        "pair_metrics": pair_metrics,
        "pair_metrics_class": pair_metrics_class,
        "tabpfn_metrics": tabpfn_metrics,
        "threshold_names": tuple(sorted(configured_names)),
        "run_ids": tuple(sorted(run_ids)),
    }



## Subset gates and validation checks



In [ ]:
def precision_recall_factors(
    rows: Sequence[Mapping[str, Any]], canonical_run: int
) -> set[int]:
    return {
        int(row["Factor"])
        for row in rows
        if int(row["run_id"]) == canonical_run
        and row.get("Provenance") == "high_precision"
    }


def f2_factors(
    rows: Sequence[Mapping[str, Any]], canonical_run: int
) -> set[int]:
    return {
        int(row["factor_id"])
        for row in rows
        if int(row["run_id"]) == canonical_run
        and bool(row.get("valid"))
        and bool(row.get("selection", {}).get("feasible"))
    }


def robust_gate(
    canonical_factor_id: int,
    direct_rows: pd.DataFrame,
    semantic_rule_lookup: Mapping[tuple[int, int, str], Mapping[str, Any]],
    other_runs: set[int],
    threshold_names: set[str],
    minimum_f2: float,
) -> tuple[bool, str]:
    rows = direct_rows[direct_rows["canonical_factor"] == canonical_factor_id]
    matched_runs = set(rows["other_run"].astype(int))
    missing_runs = sorted(other_runs - matched_runs)
    if missing_runs:
        return False, f"missing_run_match:{','.join(map(str, missing_runs))}"
    for other_run in sorted(other_runs):
        run_rows = rows[rows["other_run"].astype(int) == other_run]
        observed = set(run_rows["threshold_name"].astype(str))
        missing_thresholds = sorted(threshold_names - observed)
        if missing_thresholds:
            return False, f"missing_threshold:{other_run}:{','.join(missing_thresholds)}"
        if run_rows.groupby("threshold_name").size().max() != 1:
            return False, f"duplicate_threshold:{other_run}"
        for threshold in sorted(threshold_names):
            metric = run_rows[run_rows["threshold_name"].astype(str) == threshold].iloc[0]
            canonical_model = semantic_rule_lookup.get(
                (CANONICAL_RUN, canonical_factor_id, threshold)
            )
            other_factor = int(metric["other_factor"])
            other_model = semantic_rule_lookup.get((other_run, other_factor, threshold))
            if canonical_model is None or other_model is None:
                return False, f"missing_model:{other_run}:{threshold}"
            if not (canonical_model.get("valid") and other_model.get("valid")):
                return False, f"invalid_model:{other_run}:{threshold}"
            value = pd.to_numeric(pd.Series([metric["transfer_min_f2"]]), errors="coerce").iloc[0]
            if not np.isfinite(value) or float(value) < minimum_f2:
                return False, f"transfer_below_gate:{other_run}:{threshold}"
    return True, ""


def direct_semantic_rows(frame: pd.DataFrame, canonical_run: int) -> pd.DataFrame:
    rows = []
    for row in frame.to_dict("records"):
        factor = canonical_factor(row, canonical_run)
        other = direct_other_run(row, canonical_run)
        if factor is None or other is None:
            continue
        run_i, run_j, factor_i, factor_j = match_fields(row)
        other_factor = factor_j if run_i == canonical_run else factor_i
        rows.append(
            {
                **row,
                "canonical_factor": factor,
                "other_run": other,
                "other_factor": other_factor,
            }
        )
    return pd.DataFrame(
        rows,
        columns=list(frame.columns)
        + ["canonical_factor", "other_run", "other_factor"],
    )


def run_unit_checks() -> None:
    assert precision_recall_factors(
        [
            {"run_id": 0, "Factor": 1, "Provenance": "high_precision"},
            {"run_id": 0, "Factor": 2, "Provenance": "forced_fallback"},
        ],
        0,
    ) == {1}
    assert f2_factors(
        [
            {"run_id": 0, "factor_id": 1, "valid": True, "selection": {"feasible": False}},
            {"run_id": 0, "factor_id": 1, "valid": True, "selection": {"feasible": True}},
            {"run_id": 0, "factor_id": 2, "valid": False, "selection": {"feasible": True}},
        ],
        0,
    ) == {1}

    thresholds = {"top_10pct_positive"}
    direct = pd.DataFrame(
        [
            {
                "canonical_factor": 0,
                "other_run": 1,
                "other_factor": 4,
                "threshold_name": "top_10pct_positive",
                "transfer_min_f2": ROBUST_F2_MINIMUM,
            }
        ]
    )
    lookup = {
        (0, 0, "top_10pct_positive"): {"valid": True, "selection": {"feasible": True}},
        (1, 4, "top_10pct_positive"): {"valid": True, "selection": {"feasible": True}},
    }
    assert robust_gate(0, direct, lookup, {1}, thresholds, ROBUST_F2_MINIMUM)[0]
    assert not robust_gate(0, direct.iloc[0:0], lookup, {1}, thresholds, ROBUST_F2_MINIMUM)[0]
    invalid = dict(lookup)
    invalid[(1, 4, "top_10pct_positive")] = {"valid": False, "selection": {"feasible": True}}
    assert not robust_gate(0, direct, invalid, {1}, thresholds, ROBUST_F2_MINIMUM)[0]
    assert not robust_gate(0, direct, lookup, {1}, {"missing"}, ROBUST_F2_MINIMUM)[0]

    with tempfile.TemporaryDirectory() as temporary:
        stats = Path(temporary) / "stats"
        semantic = stats / "semantic" / "experiment"
        semantic.mkdir(parents=True)
        for name in SEMANTIC_REQUIRED:
            (semantic / name).touch()
        for index, name in enumerate(("old", "new")):
            runner = stats / "comparison" / name
            runner.mkdir(parents=True)
            for required in RUNNER_REQUIRED:
                (runner / required).touch()
            (runner / "summary.json").write_text(
                json.dumps(
                    {
                        "semantic_experiment_hash": "experiment",
                        "semantic_artifact_dir": str(semantic),
                    }
                ),
                encoding="utf-8",
            )
            os.utime(runner / "summary.json", (100 + index, 100 + index))
        selected, _, _ = discover_runner(stats)
        assert selected.name == "new"


run_unit_checks()
print("Pure validation checks passed.")



## Load selected completed run



In [ ]:
artifacts = load_artifacts(
    STATS_ROOT,
    RUNNER_DIR,
    RUNNER_HASH,
    CANONICAL_RUN,
)
RUNNER_HASH_RESOLVED = str(artifacts["summary"]["runner_hash"])
OUTPUT_DIR = STATS_ROOT / "analysis" / "robustness" / RUNNER_HASH_RESOLVED
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Runner: {artifacts['runner_dir']}")
print(f"Semantic artifacts: {artifacts['semantic_dir']}")
print(f"Output: {OUTPUT_DIR}")



## Canonical registry and overlapping factor membership



In [ ]:
canonical_activations = artifacts["activations"][CANONICAL_RUN]
n_factors = canonical_activations.shape[1]
all_factors = set(range(n_factors))
precision_factors = precision_recall_factors(
    artifacts["high_precision"], CANONICAL_RUN
)
f2_selected = f2_factors(artifacts["semantic_rules"], CANONICAL_RUN)

semantic_lookup = {
    (int(row["run_id"]), int(row["factor_id"]), str(row["target"]["name"])): row
    for row in artifacts["semantic_rules"]
}
direct_metrics = direct_semantic_rows(artifacts["pair_metrics"], CANONICAL_RUN)
other_runs = set(artifacts["run_ids"]) - {CANONICAL_RUN}
robust_reasons: dict[int, str] = {}
robust_factors: set[int] = set()
for factor_id in sorted(all_factors):
    valid, reason = robust_gate(
        factor_id,
        direct_metrics,
        semantic_lookup,
        other_runs,
        set(artifacts["threshold_names"]),
        ROBUST_F2_MINIMUM,
    )
    robust_reasons[factor_id] = reason
    if valid:
        robust_factors.add(factor_id)

fixed_subsets = {
    "all": all_factors,
    "precision_recall": precision_factors,
    "f2": f2_selected,
    "robust": robust_factors,
}
matched_runs_by_factor = (
    direct_metrics.groupby("canonical_factor")["other_run"].nunique().to_dict()
    if not direct_metrics.empty
    else {}
)
valid_thresholds_by_factor = Counter()
for factor_id in all_factors:
    valid_thresholds_by_factor[factor_id] = sum(
        bool(semantic_lookup.get((CANONICAL_RUN, factor_id, threshold), {}).get("valid"))
        and bool(
            semantic_lookup.get((CANONICAL_RUN, factor_id, threshold), {})
            .get("selection", {})
            .get("feasible")
        )
        for threshold in artifacts["threshold_names"]
    )

membership_rows = []
for factor_id in sorted(all_factors):
    flags = {
        "all": True,
        "non_dead": bool(np.any(canonical_activations[:, factor_id] > ACTIVATION_EPSILON)),
        "precision_recall": factor_id in precision_factors,
        "f2": factor_id in f2_selected,
        "robust": factor_id in robust_factors,
    }
    for subset in SUBSET_ORDER:
        membership_rows.append(
            {
                "canonical_run": CANONICAL_RUN,
                "factor_id": factor_id,
                "subset": subset,
                "selected": flags[subset],
                "globally_non_dead": flags["non_dead"],
                "matched_run_count": int(matched_runs_by_factor.get(factor_id, 0)),
                "matched_run_coverage": (
                    matched_runs_by_factor.get(factor_id, 0) / len(other_runs)
                    if other_runs
                    else np.nan
                ),
                "valid_threshold_count": int(valid_thresholds_by_factor[factor_id]),
                "required_threshold_count": len(artifacts["threshold_names"]),
                "robust_invalid_reason": robust_reasons[factor_id],
                "robust_is_posthoc": subset == "robust",
            }
        )
factor_membership = pd.DataFrame(membership_rows)
factor_membership.to_csv(OUTPUT_DIR / "factor_membership.csv", index=False)

inventory_rows = []
for subset in SUBSET_ORDER:
    selected = set(
        factor_membership.loc[
            (factor_membership["subset"] == subset) & factor_membership["selected"],
            "factor_id",
        ]
    )
    inventory_rows.append(
        {
            "subset": subset,
            "n_selected": len(selected),
            "fraction_of_all": len(selected) / n_factors if n_factors else np.nan,
            "n_globally_dead": sum(
                not np.any(canonical_activations[:, factor] > ACTIVATION_EPSILON)
                for factor in selected
            ),
            "n_matched_all_runs": sum(
                matched_runs_by_factor.get(factor, 0) == len(other_runs)
                for factor in selected
            ),
            "n_valid_all_thresholds": sum(
                valid_thresholds_by_factor[factor] == len(artifacts["threshold_names"])
                for factor in selected
            ),
        }
    )
subset_inventory = pd.DataFrame(inventory_rows)

flag_wide = factor_membership.pivot(
    index="factor_id", columns="subset", values="selected"
).fillna(False)
intersection_counts = (
    flag_wide.astype(bool)
    .value_counts()
    .rename("n_factors")
    .reset_index()
)
intersection_counts["intersection"] = intersection_counts.apply(
    lambda row: " & ".join(
        SUBSET_LABELS[name] for name in SUBSET_ORDER if bool(row[name])
    )
    or "None",
    axis=1,
)
intersection_counts.to_csv(OUTPUT_DIR / "subset_intersections.csv", index=False)
subset_inventory.to_csv(OUTPUT_DIR / "subset_inventory.csv", index=False)
display(subset_inventory)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(
    [SUBSET_LABELS[name] for name in subset_inventory["subset"]],
    subset_inventory["n_selected"],
)
axes[0].set_ylabel("Canonical factors")
axes[0].tick_params(axis="x", rotation=25)
top_intersections = intersection_counts.nlargest(12, "n_factors")
axes[1].barh(top_intersections["intersection"], top_intersections["n_factors"])
axes[1].invert_yaxis()
axes[1].set_xlabel("Canonical factors")
axes[1].set_title("Largest overlapping intersections")
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(OUTPUT_DIR / f"subset_inventory.{PLOT_FORMAT}", dpi=FIGURE_DPI, bbox_inches="tight")
if DISPLAY_PLOTS:
    plt.show()
else:
    plt.close(fig)



## Temporal robustness

Each reference year is compared with itself and all later years. Run-0
activations stay frozen. Positive-activation medians and reference non-dead masks
are fitted once on pooled reference-year rows, then reused for pooled and class
analyses. Class rows are never resampled.



In [ ]:
def safe_mean(values: Sequence[float] | np.ndarray) -> float:
    array = np.asarray(values, dtype=float)
    finite = array[np.isfinite(array)]
    return float(finite.mean()) if finite.size else np.nan


def association_vector(X: np.ndarray, activation: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    activation = np.asarray(activation, dtype=float)
    result = np.full(X.shape[1], np.nan, dtype=float)
    for column_index in range(X.shape[1]):
        column = X[:, column_index]
        valid = np.isfinite(column) & np.isfinite(activation)
        if valid.sum() < 2:
            continue
        left, right = column[valid], activation[valid]
        if np.ptp(left) == 0 or np.ptp(right) == 0:
            continue
        result[column_index] = np.corrcoef(left, right)[0, 1]
    return result


def finite_cosine(left: np.ndarray, right: np.ndarray) -> float:
    valid = np.isfinite(left) & np.isfinite(right)
    if not valid.any():
        return np.nan
    left_valid, right_valid = left[valid], right[valid]
    denominator = np.linalg.norm(left_valid) * np.linalg.norm(right_valid)
    return float(np.dot(left_valid, right_valid) / denominator) if denominator > 0 else np.nan


def positive_medians(values: np.ndarray) -> np.ndarray:
    medians = np.full(values.shape[1], np.nan, dtype=float)
    for factor_index in range(values.shape[1]):
        positive = values[:, factor_index]
        positive = positive[np.isfinite(positive) & (positive > 0)]
        if positive.size:
            medians[factor_index] = float(np.median(positive))
    return medians


def temporal_row(
    *,
    X: np.ndarray,
    activations: np.ndarray,
    years: np.ndarray,
    outcome: np.ndarray,
    reference_year: int,
    test_year: int,
    subset: str,
    fixed_factors: set[int],
    pooled_reference_active: np.ndarray,
    pooled_medians: np.ndarray,
    outcome_class: int | None,
    matched_factor_set: set[int],
) -> dict[str, Any]:
    pooled_reference_rows = years == reference_year
    reference_rows = pooled_reference_rows.copy()
    test_rows = years == test_year
    cohort = "pooled"
    if outcome_class is not None:
        reference_rows &= outcome == outcome_class
        test_rows &= outcome == outcome_class
        cohort = f"class_{outcome_class}"

    selected = set(fixed_factors)
    if subset == "non_dead":
        selected = {
            factor for factor in range(activations.shape[1])
            if pooled_reference_active[factor]
        }
    selected_array = np.asarray(sorted(selected), dtype=int)
    reference_active = np.asarray(
        [
            factor for factor in selected_array
            if pooled_reference_active[factor]
        ],
        dtype=int,
    )
    reasons: list[str] = []
    if not selected_array.size:
        reasons.append("empty_subset")
    if not reference_active.size:
        reasons.append("no_reference_active")
    if not reference_rows.any():
        reasons.append("no_reference_rows")
    if not test_rows.any():
        reasons.append("no_test_rows")

    result: dict[str, Any] = {
        "cohort": cohort,
        "outcome_class": outcome_class,
        "subset": subset,
        "reference_year": int(reference_year),
        "test_year": int(test_year),
        "temporal_distance": int(test_year - reference_year),
        "n_reference_rows": int(reference_rows.sum()),
        "n_test_rows": int(test_rows.sum()),
        "n_selected": int(selected_array.size),
        "n_reference_active": int(reference_active.size),
        "n_evaluable": 0,
        "n_evaluable_associations": 0,
        "matched_count": int(sum(factor in matched_factor_set for factor in selected_array)),
        "matched_coverage": (
            sum(factor in matched_factor_set for factor in selected_array) / selected_array.size
            if selected_array.size
            else np.nan
        ),
        "association_cosine": np.nan,
        "factor_survival": np.nan,
        "dead_fraction": np.nan,
        "stable_fraction": np.nan,
        "overused_fraction": np.nan,
        "high_activation_magnitude_ratio": np.nan,
        "active_factors_per_sample": np.nan,
        "invalid_reasons": "",
    }
    if reasons:
        result["invalid_reasons"] = "|".join(reasons)
        return result

    reference_codes = activations[reference_rows]
    test_codes = activations[test_rows]
    result["active_factors_per_sample"] = safe_mean(
        (test_codes[:, selected_array] > ACTIVATION_EPSILON).sum(axis=1)
    )

    survived = np.any(
        test_codes[:, reference_active] > ACTIVATION_EPSILON, axis=0
    )
    result["factor_survival"] = safe_mean(survived.astype(float))

    association_scores: list[float] = []
    for factor in reference_active:
        reference_association = association_vector(
            X[reference_rows], reference_codes[:, factor]
        )
        test_association = association_vector(X[test_rows], test_codes[:, factor])
        association_scores.append(finite_cosine(reference_association, test_association))
    result["n_evaluable_associations"] = int(np.isfinite(association_scores).sum())
    result["association_cosine"] = safe_mean(association_scores)
    if result["n_evaluable_associations"] == 0:
        reasons.append("no_evaluable_associations")

    count_ratios: list[float] = []
    magnitude_ratios: list[float] = []
    for factor in reference_active:
        threshold = pooled_medians[factor]
        if not np.isfinite(threshold):
            continue
        reference_high = reference_codes[:, factor] >= threshold
        test_high = test_codes[:, factor] >= threshold
        reference_count = int(reference_high.sum())
        if reference_count == 0:
            continue
        count_ratios.append(float(test_high.sum() / reference_count))
        reference_magnitude = safe_mean(reference_codes[reference_high, factor])
        test_magnitude = safe_mean(test_codes[test_high, factor])
        magnitude_ratios.append(
            test_magnitude / reference_magnitude
            if np.isfinite(test_magnitude)
            and np.isfinite(reference_magnitude)
            and reference_magnitude != 0
            else np.nan
        )

    ratio_array = np.asarray(count_ratios, dtype=float)
    result["n_evaluable"] = int(np.isfinite(ratio_array).sum())
    if result["n_evaluable"]:
        result["dead_fraction"] = safe_mean((ratio_array < 0.1).astype(float))
        result["stable_fraction"] = safe_mean(
            ((ratio_array >= 0.75) & (ratio_array <= 1.5)).astype(float)
        )
        result["overused_fraction"] = safe_mean((ratio_array > 2).astype(float))
        result["high_activation_magnitude_ratio"] = safe_mean(magnitude_ratios)
    else:
        reasons.append("no_evaluable_ratios")
    result["invalid_reasons"] = "|".join(sorted(set(reasons)))
    return result


def compute_temporal_metrics(
    X: np.ndarray,
    activations: np.ndarray,
    years: np.ndarray,
    outcome: np.ndarray,
    fixed_subsets: Mapping[str, set[int]],
    matched_factor_set: set[int],
) -> pd.DataFrame:
    records: list[dict[str, Any]] = []
    unique_years = sorted(int(year) for year in np.unique(years))
    for reference_year in unique_years:
        pooled_reference_rows = years == reference_year
        pooled_reference_codes = activations[pooled_reference_rows]
        pooled_reference_active = np.any(
            pooled_reference_codes > ACTIVATION_EPSILON, axis=0
        )
        pooled_medians = positive_medians(pooled_reference_codes)
        for test_year in unique_years:
            if test_year < reference_year:
                continue
            for subset in SUBSET_ORDER:
                fixed = (
                    set(range(activations.shape[1]))
                    if subset == "non_dead"
                    else set(fixed_subsets[subset])
                )
                records.append(
                    temporal_row(
                        X=X,
                        activations=activations,
                        years=years,
                        outcome=outcome,
                        reference_year=reference_year,
                        test_year=test_year,
                        subset=subset,
                        fixed_factors=fixed,
                        pooled_reference_active=pooled_reference_active,
                        pooled_medians=pooled_medians,
                        outcome_class=None,
                        matched_factor_set=matched_factor_set,
                    )
                )
                for class_value in OUTCOME_CLASSES:
                    records.append(
                        temporal_row(
                            X=X,
                            activations=activations,
                            years=years,
                            outcome=outcome,
                            reference_year=reference_year,
                            test_year=test_year,
                            subset=subset,
                            fixed_factors=fixed,
                            pooled_reference_active=pooled_reference_active,
                            pooled_medians=pooled_medians,
                            outcome_class=class_value,
                            matched_factor_set=matched_factor_set,
                        )
                    )
    return pd.DataFrame(records)


_empty_test = temporal_row(
    X=np.empty((0, 2)),
    activations=np.empty((0, 2)),
    years=np.asarray([], dtype=int),
    outcome=np.asarray([], dtype=int),
    reference_year=2000,
    test_year=2000,
    subset="all",
    fixed_factors=set(),
    pooled_reference_active=np.asarray([False, False]),
    pooled_medians=np.asarray([np.nan, np.nan]),
    outcome_class=None,
    matched_factor_set=set(),
)
assert np.isnan(_empty_test["association_cosine"])
assert "empty_subset" in _empty_test["invalid_reasons"]
assert np.isnan(finite_cosine(np.asarray([0.0]), np.asarray([0.0])))
_dead_reference_test = temporal_row(
    X=np.asarray([[1.0], [2.0]]),
    activations=np.asarray([[0.0], [1.0]]),
    years=np.asarray([2000, 2001]),
    outcome=np.asarray([0, 0]),
    reference_year=2000,
    test_year=2001,
    subset="all",
    fixed_factors={0},
    pooled_reference_active=np.asarray([False]),
    pooled_medians=np.asarray([np.nan]),
    outcome_class=None,
    matched_factor_set=set(),
)
assert _dead_reference_test["n_reference_active"] == 0
assert np.isnan(_dead_reference_test["high_activation_magnitude_ratio"])
_missing_class_test = temporal_row(
    X=np.asarray([[1.0], [2.0]]),
    activations=np.asarray([[1.0], [2.0]]),
    years=np.asarray([2000, 2001]),
    outcome=np.asarray([0, 0]),
    reference_year=2000,
    test_year=2001,
    subset="all",
    fixed_factors={0},
    pooled_reference_active=np.asarray([True]),
    pooled_medians=np.asarray([1.0]),
    outcome_class=1,
    matched_factor_set=set(),
)
assert "no_reference_rows" in _missing_class_test["invalid_reasons"]

matched_canonical_factors = set(direct_metrics["canonical_factor"].astype(int))
temporal_metrics = compute_temporal_metrics(
    artifacts["X"],
    canonical_activations,
    artifacts["years"],
    artifacts["outcome"],
    fixed_subsets,
    matched_canonical_factors,
)
temporal_metrics.to_csv(OUTPUT_DIR / "temporal_metrics.csv", index=False)

temporal_by_distance = (
    temporal_metrics.groupby(
        ["cohort", "outcome_class", "subset", "temporal_distance"],
        dropna=False,
    )
    .agg(
        n_comparisons=("test_year", "size"),
        n_valid_associations=("association_cosine", "count"),
        association_cosine=("association_cosine", "mean"),
        factor_survival=("factor_survival", "mean"),
        dead_fraction=("dead_fraction", "mean"),
        stable_fraction=("stable_fraction", "mean"),
        overused_fraction=("overused_fraction", "mean"),
        high_activation_magnitude_ratio=("high_activation_magnitude_ratio", "mean"),
        active_factors_per_sample=("active_factors_per_sample", "mean"),
        n_selected=("n_selected", "mean"),
        n_reference_active=("n_reference_active", "mean"),
        n_evaluable=("n_evaluable", "sum"),
        matched_coverage=("matched_coverage", "mean"),
    )
    .reset_index()
)
temporal_by_distance.to_csv(OUTPUT_DIR / "temporal_by_distance.csv", index=False)
display(temporal_by_distance.head(15))



In [ ]:
TEMPORAL_PLOT_METRICS = (
    ("association_cosine", "Association cosine"),
    ("factor_survival", "Factor survival"),
    ("dead_fraction", "Dead activation ratio"),
    ("stable_fraction", "Stable activation ratio"),
    ("overused_fraction", "Overused activation ratio"),
    ("high_activation_magnitude_ratio", "High-activation magnitude ratio"),
    ("active_factors_per_sample", "Active factors per sample"),
)


def finish_figure(fig: plt.Figure, filename: str) -> None:
    fig.tight_layout()
    if SAVE_PLOTS:
        fig.savefig(
            OUTPUT_DIR / f"{filename}.{PLOT_FORMAT}",
            dpi=FIGURE_DPI,
            bbox_inches="tight",
        )
    if DISPLAY_PLOTS:
        plt.show()
    else:
        plt.close(fig)


def plot_temporal_facets(frame: pd.DataFrame, cohort: str, filename: str) -> None:
    selected = frame[frame["cohort"] == cohort]
    fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=True)
    for axis, (metric, label) in zip(axes.flat, TEMPORAL_PLOT_METRICS):
        for subset in SUBSET_ORDER:
            rows = selected[selected["subset"] == subset].sort_values("temporal_distance")
            axis.plot(
                rows["temporal_distance"],
                rows[metric],
                marker="o",
                label=SUBSET_LABELS[subset],
            )
        axis.set_title(label)
        axis.set_xlabel("Temporal distance (years)")
    for axis in axes.flat[len(TEMPORAL_PLOT_METRICS):]:
        axis.set_visible(False)
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(f"Temporal robustness: {cohort.replace('_', ' ')}", y=1.01)
    finish_figure(fig, filename)


for cohort_name in ("pooled", "class_0", "class_1"):
    plot_temporal_facets(
        temporal_by_distance,
        cohort_name,
        f"temporal_{cohort_name}",
    )



## Drift versus persisted model performance

`concept_drift_full.pkl` is preferred because it contains reference/test-year
pairs. Otherwise persisted `tabpfn_metrics.json` is plotted on its actual calendar
years. No missing reference-year model value is synthesized.



In [ ]:
def normalize_from_baseline(values: pd.Series, baseline: float) -> pd.Series:
    if np.isfinite(baseline) and baseline != 0:
        return values.astype(float) / baseline
    return pd.Series(np.nan, index=values.index, dtype=float)


def plot_drift_model_comparison(subset: str) -> str:
    concept_path = STATS_ROOT / "concept_drift_full.pkl"
    pooled = temporal_metrics[
        (temporal_metrics["cohort"] == "pooled")
        & (temporal_metrics["subset"] == subset)
    ].copy()
    fig, axis = plt.subplots(figsize=(9, 5))
    source: str
    if concept_path.is_file():
        legacy = pd.read_pickle(concept_path)
        require_columns(legacy, ("train_year", "test_year"), "concept_drift_full.pkl")
        legacy = legacy.copy()
        legacy["temporal_distance"] = legacy["test_year"] - legacy["train_year"]
        model_columns = [
            column
            for column in ("f1_macro", "f1_pos", "accuracy", "roc_auc_score")
            if column in legacy.columns
        ]
        concept_agg = pooled.groupby("temporal_distance").agg(
            association_cosine=("association_cosine", "mean"),
            factor_survival=("factor_survival", "mean"),
        )
        for metric, label in (
            ("association_cosine", "Association cosine"),
            ("factor_survival", "Factor survival"),
        ):
            baseline = concept_agg.loc[0, metric] if 0 in concept_agg.index else np.nan
            axis.plot(
                concept_agg.index,
                normalize_from_baseline(concept_agg[metric], baseline),
                marker="o",
                label=label,
            )
        model_agg = legacy.groupby("temporal_distance")[model_columns].mean()
        for metric in model_columns:
            baseline = model_agg.loc[0, metric] if 0 in model_agg.index else np.nan
            axis.plot(
                model_agg.index,
                normalize_from_baseline(model_agg[metric], baseline),
                marker="s",
                linestyle="--",
                label=f"Model {metric}",
            )
        axis.set_xlabel("Temporal distance (years)")
        source = "stats/concept_drift_full.pkl"
    else:
        model = artifacts["tabpfn_metrics"].copy()
        concept_agg = pooled.groupby("test_year").agg(
            association_cosine=("association_cosine", "mean"),
            factor_survival=("factor_survival", "mean"),
        )
        for metric, label in (
            ("association_cosine", "Association cosine"),
            ("factor_survival", "Factor survival"),
        ):
            finite = concept_agg[metric].dropna()
            baseline = finite.iloc[0] if len(finite) else np.nan
            axis.plot(
                concept_agg.index,
                normalize_from_baseline(concept_agg[metric], baseline),
                marker="o",
                label=label,
            )
        if not model.empty:
            require_columns(model, ("year",), "tabpfn_metrics.json")
            model_columns = [
                column for column in ("f1_macro", "f1_pos") if column in model.columns
            ]
            for metric in model_columns:
                ordered = model.sort_values("year")
                finite = ordered[metric].dropna()
                baseline = finite.iloc[0] if len(finite) else np.nan
                axis.plot(
                    ordered["year"],
                    normalize_from_baseline(ordered[metric], baseline),
                    marker="s",
                    linestyle="--",
                    label=f"Model {metric}",
                )
            source = str(artifacts["runner_dir"] / "tabpfn_metrics.json")
        else:
            source = "unavailable: tabpfn_metrics.json contains no persisted rows"
        axis.set_xlabel("Persisted calendar year")
    axis.axhline(1.0, color="black", linewidth=1, alpha=0.45)
    axis.set_ylabel("Value / first persisted baseline")
    axis.set_title(f"Drift and model performance: {SUBSET_LABELS[subset]}")
    axis.legend(fontsize=8, ncol=2)
    finish_figure(fig, f"drift_model_{subset}")
    return source


model_performance_source = ""
for subset_name in SUBSET_ORDER:
    model_performance_source = plot_drift_model_comparison(subset_name)



## Pooled semantic transfer

Only direct run-0 pairs are compared. Subsets overlap, so one pair/threshold may
appear in several subset rows. Robust membership is explicitly post-hoc because
it uses held-out pooled transfer.



In [ ]:
def add_semantic_derived_columns(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for column in (
        "i_to_j_precision", "i_to_j_recall", "i_to_j_f2",
        "j_to_i_precision", "j_to_i_recall", "j_to_i_f2",
        "transfer_mean_f2", "transfer_min_f2", "target_cohort_jaccard",
        "selected_cohort_jaccard", "cos_sim",
    ):
        if column in result:
            result[column] = pd.to_numeric(result[column], errors="coerce")
    result["precision_mean"] = result[["i_to_j_precision", "j_to_i_precision"]].mean(axis=1)
    result["recall_mean"] = result[["i_to_j_recall", "j_to_i_recall"]].mean(axis=1)
    result["directional_f2_asymmetry"] = (
        result["i_to_j_f2"] - result["j_to_i_f2"]
    ).abs()
    stability_keys = ["pair_id", "canonical_factor", "other_run"]
    if "class_value" in result.columns:
        stability_keys.append("class_value")
    stability = (
        result.groupby(stability_keys, dropna=False)["transfer_mean_f2"]
        .agg(threshold_f2_min="min", threshold_f2_max="max")
        .reset_index()
    )
    stability["threshold_f2_range"] = (
        stability["threshold_f2_max"] - stability["threshold_f2_min"]
    )
    return result.merge(
        stability[
            stability_keys + ["threshold_f2_range"]
        ],
        on=stability_keys,
        how="left",
        validate="many_to_one",
    )


def expand_by_subset(
    direct: pd.DataFrame,
    membership: pd.DataFrame,
) -> pd.DataFrame:
    selected_membership = membership[membership["selected"]][
        ["factor_id", "subset"]
    ].rename(columns={"factor_id": "canonical_factor"})
    return direct.merge(
        selected_membership,
        on="canonical_factor",
        how="inner",
        validate="many_to_many",
    )


semantic_metrics = expand_by_subset(
    add_semantic_derived_columns(direct_metrics),
    factor_membership,
)
semantic_metrics["robust_is_posthoc"] = semantic_metrics["subset"] == "robust"
semantic_metrics.to_csv(OUTPUT_DIR / "semantic_metrics.csv", index=False)

semantic_summary = (
    semantic_metrics.groupby("subset")
    .agg(
        n_rows=("pair_id", "size"),
        n_pairs=("pair_id", "nunique"),
        n_factors=("canonical_factor", "nunique"),
        precision_mean=("precision_mean", "mean"),
        recall_mean=("recall_mean", "mean"),
        transfer_f2_mean=("transfer_mean_f2", "mean"),
        transfer_f2_min_mean=("transfer_min_f2", "mean"),
        transfer_f2_worst=("transfer_min_f2", "min"),
        directional_f2_asymmetry=("directional_f2_asymmetry", "mean"),
        threshold_f2_range=("threshold_f2_range", "mean"),
        target_cohort_jaccard=("target_cohort_jaccard", "mean"),
        selected_cohort_jaccard=("selected_cohort_jaccard", "mean"),
        cosine_similarity=("cos_sim", "mean"),
    )
    .reindex(SUBSET_ORDER)
    .reset_index()
)
semantic_summary.to_csv(OUTPUT_DIR / "semantic_summary.csv", index=False)
display(semantic_summary)


def subset_boxplots(
    frame: pd.DataFrame,
    metrics: Sequence[tuple[str, str]],
    filename: str,
    title: str,
) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    for axis, (metric, label) in zip(axes.flat, metrics):
        values = [
            frame.loc[frame["subset"] == subset, metric].dropna().to_numpy()
            for subset in SUBSET_ORDER
        ]
        axis.boxplot(
            values,
            tick_labels=[SUBSET_LABELS[subset] for subset in SUBSET_ORDER],
            showfliers=False,
        )
        axis.set_title(label)
        axis.tick_params(axis="x", rotation=25)
    fig.suptitle(title, y=1.01)
    finish_figure(fig, filename)


subset_boxplots(
    semantic_metrics,
    (
        ("precision_mean", "Mean directional precision"),
        ("recall_mean", "Mean directional recall"),
        ("transfer_mean_f2", "Mean transfer F2"),
        ("transfer_min_f2", "Minimum directional F2"),
    ),
    "semantic_pooled_core",
    "Pooled semantic transfer",
)
subset_boxplots(
    semantic_metrics,
    (
        ("directional_f2_asymmetry", "Directional F2 asymmetry"),
        ("threshold_f2_range", "Threshold F2 range"),
        ("target_cohort_jaccard", "Target-cohort Jaccard"),
        ("selected_cohort_jaccard", "Selected-cohort Jaccard"),
    ),
    "semantic_pooled_stability",
    "Pooled semantic stability and cohorts",
)

fig, axes = plt.subplots(1, len(SUBSET_ORDER), figsize=(18, 3.8), sharex=True, sharey=True)
for axis, subset in zip(axes, SUBSET_ORDER):
    rows = semantic_metrics[semantic_metrics["subset"] == subset]
    scatter = axis.scatter(
        rows["cos_sim"],
        rows["transfer_mean_f2"],
        c=rows["positive_fraction"],
        cmap="viridis",
        alpha=0.75,
    )
    axis.set_title(SUBSET_LABELS[subset])
    axis.set_xlabel("Cosine similarity")
axes[0].set_ylabel("Mean transfer F2")
if len(semantic_metrics):
    fig.colorbar(scatter, ax=axes, label="Positive fraction", shrink=0.8)
finish_figure(fig, "geometry_vs_transfer")



## Outcome-stratified semantic transfer

Rules, cutoffs, and factor subsets remain frozen. Class rows only evaluate the
held-out final split. Invalid or unsupported cohorts remain explicit.



In [ ]:
direct_class_metrics = direct_semantic_rows(
    artifacts["pair_metrics_class"], CANONICAL_RUN
)
semantic_metrics_class = expand_by_subset(
    add_semantic_derived_columns(direct_class_metrics),
    factor_membership,
)
semantic_metrics_class["class_value"] = pd.to_numeric(
    semantic_metrics_class["class_value"], errors="coerce"
)
semantic_metrics_class["class_valid"] = (
    semantic_metrics_class["class_valid"]
    .astype(str)
    .str.lower()
    .map({"true": True, "false": False})
    .fillna(False)
)
semantic_metrics_class["robust_is_posthoc"] = (
    semantic_metrics_class["subset"] == "robust"
)
semantic_metrics_class.to_csv(
    OUTPUT_DIR / "semantic_metrics_by_class.csv", index=False
)

semantic_summary_class = (
    semantic_metrics_class.groupby(["subset", "class_value"], dropna=False)
    .agg(
        n_rows=("pair_id", "size"),
        n_valid=("class_valid", "sum"),
        n_invalid=("class_valid", lambda values: int((~values).sum())),
        n_samples_min=("class_n_samples", "min"),
        n_samples_max=("class_n_samples", "max"),
        n_positive_i_min=("class_n_positive_i", "min"),
        n_positive_i_max=("class_n_positive_i", "max"),
        n_positive_j_min=("class_n_positive_j", "min"),
        n_positive_j_max=("class_n_positive_j", "max"),
        precision_mean=("precision_mean", "mean"),
        recall_mean=("recall_mean", "mean"),
        transfer_f2_mean=("transfer_mean_f2", "mean"),
        transfer_f2_min_mean=("transfer_min_f2", "mean"),
        directional_f2_asymmetry=("directional_f2_asymmetry", "mean"),
        threshold_f2_range=("threshold_f2_range", "mean"),
        target_cohort_jaccard=("target_cohort_jaccard", "mean"),
        selected_cohort_jaccard=("selected_cohort_jaccard", "mean"),
    )
    .reset_index()
)
semantic_summary_class.to_csv(
    OUTPUT_DIR / "semantic_summary_by_class.csv", index=False
)

gap_keys = ["subset", "pair_id", "canonical_factor", "other_run", "threshold_name"]
class_zero = semantic_metrics_class[
    semantic_metrics_class["class_value"] == 0
].copy()
class_one = semantic_metrics_class[
    semantic_metrics_class["class_value"] == 1
].copy()
gap_metrics = (
    class_one.merge(
        class_zero,
        on=gap_keys,
        how="outer",
        suffixes=("_class1", "_class0"),
        indicator=True,
        validate="one_to_one",
    )
)
for metric in (
    "precision_mean",
    "recall_mean",
    "transfer_mean_f2",
    "transfer_min_f2",
    "directional_f2_asymmetry",
    "target_cohort_jaccard",
    "selected_cohort_jaccard",
):
    gap_metrics[f"{metric}_gap_class1_minus_class0"] = (
        gap_metrics[f"{metric}_class1"] - gap_metrics[f"{metric}_class0"]
    )
gap_metrics.to_csv(OUTPUT_DIR / "semantic_class_gaps.csv", index=False)
display(semantic_summary_class)


fig, axes = plt.subplots(2, 2, figsize=(13, 9))
class_plot_metrics = (
    ("precision_mean", "Mean directional precision"),
    ("recall_mean", "Mean directional recall"),
    ("transfer_mean_f2", "Mean transfer F2"),
    ("transfer_min_f2", "Minimum directional F2"),
)
positions = np.arange(len(SUBSET_ORDER))
width = 0.36
for axis, (metric, label) in zip(axes.flat, class_plot_metrics):
    for offset, class_value in ((-width / 2, 0), (width / 2, 1)):
        summary = (
            semantic_metrics_class[
                semantic_metrics_class["class_value"] == class_value
            ]
            .groupby("subset")[metric]
            .mean()
            .reindex(SUBSET_ORDER)
        )
        axis.bar(
            positions + offset,
            summary,
            width=width,
            label=f"DEATH={class_value}",
        )
    axis.set_title(label)
    axis.set_xticks(positions, [SUBSET_LABELS[name] for name in SUBSET_ORDER], rotation=25)
axes[0, 0].legend()
fig.suptitle("Class-stratified semantic transfer", y=1.01)
finish_figure(fig, "semantic_by_class")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
class_stability_metrics = (
    ("directional_f2_asymmetry", "Directional F2 asymmetry"),
    ("threshold_f2_range", "Threshold F2 range"),
    ("target_cohort_jaccard", "Target-cohort Jaccard"),
    ("selected_cohort_jaccard", "Selected-cohort Jaccard"),
)
for axis, (metric, label) in zip(axes.flat, class_stability_metrics):
    for offset, class_value in ((-width / 2, 0), (width / 2, 1)):
        summary = (
            semantic_metrics_class[
                semantic_metrics_class["class_value"] == class_value
            ]
            .groupby("subset")[metric]
            .mean()
            .reindex(SUBSET_ORDER)
        )
        axis.bar(
            positions + offset,
            summary,
            width=width,
            label=f"DEATH={class_value}",
        )
    axis.set_title(label)
    axis.set_xticks(
        positions,
        [SUBSET_LABELS[name] for name in SUBSET_ORDER],
        rotation=25,
    )
axes[0, 0].legend()
fig.suptitle("Class-stratified semantic stability and cohorts", y=1.01)
finish_figure(fig, "semantic_by_class_stability")

fig, axes = plt.subplots(
    len(OUTCOME_CLASSES),
    len(SUBSET_ORDER),
    figsize=(18, 7),
    sharex=True,
    sharey=True,
    squeeze=False,
)
for row_index, class_value in enumerate(OUTCOME_CLASSES):
    for column_index, subset in enumerate(SUBSET_ORDER):
        axis = axes[row_index, column_index]
        rows = semantic_metrics_class[
            (semantic_metrics_class["class_value"] == class_value)
            & (semantic_metrics_class["subset"] == subset)
        ]
        axis.scatter(
            rows["cos_sim"],
            rows["transfer_mean_f2"],
            c=rows["positive_fraction"],
            cmap="viridis",
            alpha=0.75,
        )
        if row_index == 0:
            axis.set_title(SUBSET_LABELS[subset])
        if column_index == 0:
            axis.set_ylabel(f"DEATH={class_value}\nMean transfer F2")
        if row_index == len(OUTCOME_CLASSES) - 1:
            axis.set_xlabel("Cosine similarity")
fig.suptitle("Geometry versus class-stratified transfer", y=1.01)
finish_figure(fig, "geometry_vs_transfer_by_class")

gap_columns = [
    column for column in gap_metrics
    if column.endswith("_gap_class1_minus_class0")
]
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for axis, metric in zip(axes.flat, gap_columns[:4]):
    values = (
        gap_metrics.groupby("subset")[metric]
        .mean()
        .reindex(SUBSET_ORDER)
    )
    axis.bar(
        [SUBSET_LABELS[name] for name in SUBSET_ORDER],
        values,
    )
    axis.axhline(0, color="black", linewidth=1)
    axis.set_title(metric.removesuffix("_gap_class1_minus_class0").replace("_", " "))
    axis.tick_params(axis="x", rotation=25)
fig.suptitle("Class 1 minus class 0 semantic gaps", y=1.01)
finish_figure(fig, "semantic_class_gaps")

remaining_gap_columns = gap_columns[4:]
if remaining_gap_columns:
    n_columns = 2
    n_rows = math.ceil(len(remaining_gap_columns) / n_columns)
    fig, axes = plt.subplots(
        n_rows,
        n_columns,
        figsize=(13, 4.2 * n_rows),
        squeeze=False,
    )
    for axis, metric in zip(axes.flat, remaining_gap_columns):
        values = (
            gap_metrics.groupby("subset")[metric]
            .mean()
            .reindex(SUBSET_ORDER)
        )
        axis.bar(
            [SUBSET_LABELS[name] for name in SUBSET_ORDER],
            values,
        )
        axis.axhline(0, color="black", linewidth=1)
        axis.set_title(
            metric.removesuffix("_gap_class1_minus_class0").replace("_", " ")
        )
        axis.tick_params(axis="x", rotation=25)
    for axis in axes.flat[len(remaining_gap_columns):]:
        axis.set_visible(False)
    fig.suptitle("Additional class 1 minus class 0 gaps", y=1.01)
    finish_figure(fig, "semantic_class_gaps_additional")

invalid_reasons = (
    semantic_metrics_class.assign(
        class_reasons=semantic_metrics_class["class_reasons"].fillna("").replace("", "valid")
    )
    .groupby(["subset", "class_value", "class_reasons"])
    .size()
    .rename("n_rows")
    .reset_index()
)
invalid_reasons.to_csv(OUTPUT_DIR / "semantic_class_support_reasons.csv", index=False)

fig, axis = plt.subplots(figsize=(10, 5))
invalid_counts = (
    semantic_summary_class.pivot(
        index="subset", columns="class_value", values="n_invalid"
    )
    .reindex(SUBSET_ORDER)
    .fillna(0)
)
invalid_counts.plot.bar(ax=axis)
axis.set_xticklabels(
    [SUBSET_LABELS[name] for name in SUBSET_ORDER], rotation=25
)
axis.set_ylabel("Invalid pair/threshold rows")
axis.legend(title="DEATH")
finish_figure(fig, "semantic_class_invalid_counts")

fig, axis = plt.subplots(figsize=(10, 5))
support = (
    semantic_summary_class.pivot(
        index="subset", columns="class_value", values="n_samples_min"
    )
    .reindex(SUBSET_ORDER)
    .fillna(0)
)
support.plot.bar(ax=axis)
axis.set_xticklabels(
    [SUBSET_LABELS[name] for name in SUBSET_ORDER], rotation=25
)
axis.set_ylabel("Minimum persisted class support")
axis.legend(title="DEATH")
finish_figure(fig, "semantic_class_support")



## Compact analysis manifest



In [ ]:
temporal_reason_counts = (
    temporal_metrics.assign(
        invalid_reasons=temporal_metrics["invalid_reasons"].replace("", "valid")
    )["invalid_reasons"]
    .value_counts()
    .to_dict()
)
analysis_manifest = {
    "runner_hash": RUNNER_HASH_RESOLVED,
    "canonical_run": CANONICAL_RUN,
    "activation_epsilon": ACTIVATION_EPSILON,
    "robust_f2_minimum": ROBUST_F2_MINIMUM,
    "robust_subset_is_posthoc": True,
    "source_runner_dir": str(artifacts["runner_dir"]),
    "source_semantic_dir": str(artifacts["semantic_dir"]),
    "output_dir": str(OUTPUT_DIR),
    "model_performance_source": model_performance_source,
    "n_rows": len(artifacts["X"]),
    "n_factors": n_factors,
    "run_ids": list(artifacts["run_ids"]),
    "threshold_names": list(artifacts["threshold_names"]),
    "subset_counts": {
        row["subset"]: int(row["n_selected"])
        for row in inventory_rows
    },
    "temporal_invalid_reason_counts": {
        str(key): int(value) for key, value in temporal_reason_counts.items()
    },
    "semantic_class_invalid_rows": int(
        (~semantic_metrics_class["class_valid"]).sum()
    ),
    "source_artifacts_modified": False,
}
with (OUTPUT_DIR / "analysis_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(analysis_manifest, handle, indent=2, sort_keys=True, allow_nan=False)

print("Analysis complete.")
print(f"Tables and plots: {OUTPUT_DIR}")
display(pd.DataFrame([analysis_manifest]).T.rename(columns={0: "value"}))
